# prototype

# Import libraries

In [ ]:
import ee
import pandas as pd
import geopandas as gpd
import numpy as np
import duckdb
import geemap
import os

print("📊 Environment successfully provisioned with Geopandas and Parquet extensions.")

## Authenticate Google Earth Engine (GEE)

In [ ]:
try:
    # Trigger an internal GEE call to test if credentials already exist
    ee.Initialize()
    print("✅ Google Earth Engine successfully initialized with existing credentials.")
except Exception as e:
    # If it fails, trigger the interactive authentication link
    print("🔑 Credentials not found or expired. Requesting authentication...")
    ee.Authenticate()
    ee.Initialize()
    print("✅ Google Earth Engine successfully authenticated and initialized.")

## Candidate Sites, Direct Ingestion & Quartile Pipeline

In [ ]:
# 1. Configuration Dictionary (Using raw Longitude/Latitude decimals!)
ASEAN_BOUNDS = {
    "Malaysia": [99.6, 0.8, 119.3, 7.5],       # [west, south, east, north]
    "Indonesia": [95.0, -11.0, 141.0, 6.0],
    "Vietnam": [102.1, 8.5, 109.5, 23.4],
    "Philippines": [116.9, 4.6, 126.6, 19.5]
}

selected_region = "Philippines"
bounds = ASEAN_BOUNDS[selected_region]

# Map to standard coordinate names
min_lon, min_lat, max_lon, max_lat = bounds

print(f"🌍 Querying {selected_region} bounds directly...")
print(f"Longitude: {min_lon} to {max_lon} | Latitude: {min_lat} to {max_lat}")

# 2. DuckDB over HTTP 
DATA_URL = 'https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2023/quarter=4/2023-10-01_performance_mobile_tiles.parquet'

# query tile_x as Longitude and tile_y as Latitude.
# pull Ookla's native 'quadkey' to use as site_id
query = f"""
    SELECT 
        quadkey AS site_id,
        tile_x AS longitude, 
        tile_y AS latitude, 
        avg_d_kbps AS download_kbps, 
        tests, 
        devices
    FROM read_parquet('{DATA_URL}')
    WHERE tile_x >= {min_lon} AND tile_x <= {max_lon}
      AND tile_y >= {min_lat} AND tile_y <= {max_lat}
"""

print("📥 Streaming Ookla data via DuckDB...")
try:
    my_ookla = duckdb.query(query).df()
    
    if len(my_ookla) == 0:
        print("⚠️ Query returned 0 rows.")
    else:
        print(f"✅ Successfully isolated {len(my_ookla):,} performance tiles.")
        
        # 3. Quartile Analysis
        q1_threshold = my_ookla['download_kbps'].quantile(0.25)
        print(f"📉 Q1 Threshold: {q1_threshold:,.2f} kbps")
        
        underserved_sites = my_ookla[my_ookla['download_kbps'] <= q1_threshold].copy()
        print(f"🎯 Target Acquired: {len(underserved_sites):,} underserved sites.")

        
    # Convert Pandas DataFrame into a spatially-aware GeoDataFrame
    gdf_sites = gpd.GeoDataFrame(
        underserved_sites,
        geometry=gpd.points_from_xy(underserved_sites.longitude, underserved_sites.latitude),
        crs="EPSG:4326"
    )

    TARGET_COUNTRY='Philippines'
    # Load custom GeoJSON and isolate the target country's shape dynamically
    asean_gdf = gpd.read_file("../data/Asean.geojson")
    target_shape = asean_gdf[asean_gdf['Country'] == TARGET_COUNTRY]

    # Perform a spatial join (keeps only the points that fall strictly INSIDE the shape)
    clipped_gdf = gpd.sjoin(gdf_sites, target_shape, predicate='within')

    #   Convert back to a clean Pandas DataFrame and drop the temporary spatial metadata
    underserved_sites = pd.DataFrame(clipped_gdf).drop(
        columns=['geometry', 'index_right', 'OBJECTID', 'Country', 'Flag'],
        errors='ignore'
    )

    print(f"✅ Spatial clip complete! New site count: {len(underserved_sites)}")

except Exception as e:
    print(f"⚠️ Pipeline Error: {e}")

## Multi-Band GeoAI Feature Composite

In [ ]:
# 1. Define the Region of Interest (Malaysia Bounding Box)
# This prevents GEE from processing global data
malaysia_roi = ee.Geometry.BBox(99.6, 0.8, 119.3, 7.5)

# 2. Get VIIRS Nighttime Lights (Proxy for Off-Grid Likelihood)
# Using the V22 annual composite as requested in your DRM
viirs = ee.ImageCollection('NOAA/VIIRS/DNB/ANNUAL_V22') \
    .filterDate('2022-01-01', '2023-01-01') \
    .select('average') \
    .median() \
    .rename('night_radiance')

# 3. Get ERA5-Land Solar Radiation (Proxy for Solar Viability)
# Filtering a recent year and taking the mean over the months
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
    .filterDate('2023-01-01', '2024-01-01') \
    .select('surface_solar_radiation_downwards_sum') \
    .mean() \
    .rename('solar_radiation')

# 4. Get WorldPop 2020 (Population dependent on site)
# Filtered specifically for the 2020 modelled surface as per your spec
worldpop = ee.ImageCollection('WorldPop/GP/100m/pop') \
    .filterDate('2020-01-01', '2021-01-01') \
    .select('population')\
    .mean() \
    .rename('population')

# 5. Get NASADEM (Elevation and derived Slope for logistics difficulty)
# NASADEM is a single Image, not an ImageCollection, so no date filter is needed
nasadem = ee.Image('NASA/NASADEM_HGT/001')
elevation = nasadem.select('elevation').rename('elevation')
slope = ee.Terrain.slope(elevation).rename('slope')

# 6. Get GPM IMERG Rainfall (Solar yield derating and monsoon resilience)
rainfall = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
    .filterDate('2023-01-01', '2024-01-01') \
    .select('precipitation') \
    .mean() \
    .rename('rainfall')

# 7. Stack ALL layers into a single multi-band Image
esg_composite = ee.Image.cat([
    viirs, 
    era5, 
    worldpop, 
    elevation, 
    slope, 
    rainfall
])

print("✅ Full GEE Composite Image Built with 6 bands. Ready for sampling.")

## Extract GEE to dataframe

In [ ]:
def extract_gee_to_dataframe(latitudes, longitudes, site_ids, image_stack, roi):
    """
    Takes a list of coordinates and ids, buffers them to 5.5km, reduces the GEE 
    image stack over those buffers, and returns a local Pandas DataFrame.
    """
    # 1. Create GEE point geometries with forced native types
    features = [
        ee.Feature(ee.Geometry.Point(float(lon), float(lat)), {'site_id': int(sid)})
        for lon, lat, sid in zip(longitudes, latitudes, site_ids)
    ]
    fc = ee.FeatureCollection(features)

    # 2. Buffer the points to create a 5.5km geometry around the centroid (2750m radius)
    buffered_fc = fc.map(lambda feature: feature.buffer(2750))

    # 3. SAFETY FILTER: Drop any points that fall outside the image stack's footprint
    safe_fc = buffered_fc.filterBounds(image_stack.geometry())

    # 4. Combine the mean and sum reducers
    combined_reducer = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.sum(),
        sharedInputs=True
    ) 

    # 5. Apply the combined reducer with an explicit projection enforcement (EPSG:4326)
    # This forces GEE to handle coordinates uniformly without crashing on CRS mismatches
    reduced = image_stack.reduceRegions(
        collection=safe_fc,
        reducer=combined_reducer,  
        scale=500,
        crs='EPSG:4326'
    )
    
    # 6. Pull the data to local machine with error tracing
    try:
        data = reduced.getInfo()
    except Exception as e:
        print(f"❌ GEE Server-Side Error Caught: {e}")
        raise e
    
    # 7. Parse the JSON response into a clean Pandas DataFrame
    records = []
    for feature in data.get('features', []):
        props = feature.get('properties', {})
        records.append(props)
        
    return pd.DataFrame(records)

# --- EXECUTION & EXTRACTION ---

# For initial testing, let's take a random sample of 20 underserved sites so GEE 
# doesn't time out while you are prototyping. (Remove .sample(20) for the final run).
test_cohort = underserved_sites.sample(20, random_state=42).copy()

# 1. Create a bulletproof Unique ID for merging
#OLDCODE :test_cohort['site_id'] = range(len(test_cohort))

# 1.Reset the index cleanly to establish a safe, sequential execution ID 
# without clashing with structural column names
test_cohort = test_cohort.reset_index(drop=True)
test_cohort['site_id'] = test_cohort.index

# ---  EXTRACT DATA FROM GEE ---
target_lats = test_cohort['latitude'].tolist()
target_lons = test_cohort['longitude'].tolist()
target_ids = test_cohort['site_id'].tolist()

print("⏳ Pinging Google Earth Engine for environmental data extraction...")
df_energy = extract_gee_to_dataframe(target_lats, target_lons, target_ids, esg_composite, malaysia_roi)
print("✅ Extraction complete.")
# -----------------------------------------------

# 2. check if GEE actually returned data
print(f"✅ GEE returned {len(df_energy)} records.")

if len(df_energy) == 0:
    print("⚠️ WARNING: df_energy is empty. Check if your points are outside the Region of Interest.")

# 3. Combine the dataframes using the rounded coordinates
master_esg_df = pd.merge(
    test_cohort, 
    df_energy, 
    on='site_id', 
    how='inner'
)

# --- POST-PROCESSING DATA FIXES ---

# Scale the environmental data. Convert solar radiation from Joules to Megajoules (MJ/m²)
if 'solar_radiation_mean' in master_esg_df.columns:
    master_esg_df['solar_radiation_mj'] = master_esg_df['solar_radiation_mean'] / 1000000

# Clean up redundant columns
columns_to_drop = [
    'site_id',
    'elevation_sum',
    'population_mean', 
    'night_radiance_sum', 
    'rainfall_sum',
    'slope_sum', 
    'solar_radiation_mean',
    'solar_radiation_sum'
]
# Clean up the temporary join columns
master_esg_df = master_esg_df.drop(columns=columns_to_drop, errors= 'ignore')

# Rename the final columns 
master_esg_df = master_esg_df.rename(columns={
    'elevation_mean': 'elevation_m',
    'slope_mean': 'slope_degrees',
    'rainfall_mean': 'rainfall_mm_hr',
    'night_radiance_mean': 'night_radiance_nw_cm2_sr', 
    'population_sum': 'population_total'
})

# --- VIEWING RESULTS ---
print(f"📊 Master Dataset Rows After Merge: {len(master_esg_df)}")

if len(master_esg_df) > 0:
    print("\n📊 Master Dataset Head:")
    display(master_esg_df.head())
    print("\n")
    # View a random sample of 5 sites (or less if the dataset is smaller than 5)
    print("Random Sample of 5 Sites")
    sample_size = min(5, len(master_esg_df))
    display(master_esg_df.sample(sample_size))
    print("\n")
    # Check for any null values 
    print("Check for null values : ")
    display(master_esg_df.info())
    print("\n")
    # Look at the statistical spread
    print("Statistical table")
    display(master_esg_df.describe())
    print("\n")
else:
    print("❌ Merge resulted in 0 rows. Check coordinate matching.")


## Parquet Packaging for Streamlit Handoff

In [ ]:
OUTPUT_DIR = '../data'
OUTPUT_FILE_NAME = 'jendela_phase2_esg_matrix.parquet'
full_output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_NAME)

# Ensure the data directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Clean structural check
display(master_esg_df.head())
# Save to ultra-compressed analytics format
master_esg_df.to_parquet(full_output_path, compression='snappy', index=False)

print(f"\n💾 Handoff file created cleanly: '{OUTPUT_FILE_NAME}'")
print("📥 Ready to commit to git/remote staging repository for dashboard deployment!")

## Map

In [ ]:

import ee

# 1. Initialize an interactive map centered on Malaysia
Map = geemap.Map(center=[4.21, 101.97], zoom=6)

# 2. Add a basemap (optional, but satellite view is helpful for checking terrain)
Map.add_basemap('SATELLITE')

# 3. Rebuild the GEE feature collection from your test_cohort
features = [
    ee.Feature(ee.Geometry.Point(lon, lat), {'site_id': sid})
    for lon, lat, sid in zip(target_lons, target_lats, target_ids)
]
points_fc = ee.FeatureCollection(features)
buffers_fc = points_fc.map(lambda f: f.buffer(2750))

# 4. Add the GEE layers to the map
# Buffers in semi-transparent blue, points in solid red
Map.addLayer(buffers_fc, {'color': 'blue'}, '5.5km Analysis Buffers', opacity=0.4)
Map.addLayer(points_fc, {'color': 'red'}, 'Underserved Sites (Ookla)')

# 5. Render the map inside VS Code
Map